In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
import rasterio


csv_file_path = 'final_drone_pm_data.csv'
map_file_path = 'iitgoa_map5_modified.tif'
output_filename = 'pm_concentration_heatmap_v2.png'

lat_col = 'Latitude'
lon_col = 'Longitude'
pm_col_to_plot = 'PM2.5'

grid_resolution = 500j
interpolation_method = 'linear'
cmap = 'jet'
alpha = 0.6



def create_pm_heatmap():
    print("--- Starting Heatmap Generation (Debug Mode) ---")


    try:
        df = pd.read_csv(csv_file_path)
    except FileNotFoundError:
        print(f"Error: The file '{csv_file_path}' was not found.")
        return


    print("\n[DIAGNOSTIC 1: DataFrame Info]")
    print("First 5 rows of your data:")
    print(df.head())
    print("\nColumn names and data types:")
    df.info()
    print("-" * 30)

    if pm_col_to_plot not in df.columns:
        print(f"\n\n!!! CRITICAL ERROR !!!")
        print(f"The column '{pm_col_to_plot}' was not found in your CSV file.")
        print(f"Available columns are: {list(df.columns)}")
        print("Please correct the 'pm_col_to_plot' variable in the script and run again.")
        return


    print("\n[DIAGNOSTIC 2: Statistics for the selected PM column]")
    print(f"Descriptive statistics for the '{pm_col_to_plot}' column:")
    print(df[pm_col_to_plot].describe())
    print("-" * 30)



    lats = df[lat_col].values
    lons = df[lon_col].values
    pm_values = df[pm_col_to_plot].values


    try:
        with rasterio.open(map_file_path) as raster:
            map_image = raster.read()
            map_extent = [raster.bounds.left, raster.bounds.right,
                          raster.bounds.bottom, raster.bounds.top]
            if map_image.ndim == 3 and map_image.shape[0] in [3, 4]:
                 map_image = np.moveaxis(map_image, 0, -1)
    except FileNotFoundError:
        print(f"Error: The map file '{map_file_path}' was not found.")
        return


    grid_x, grid_y = np.mgrid[map_extent[0]:map_extent[1]:grid_resolution,
                              map_extent[2]:map_extent[3]:grid_resolution]
    grid_z = griddata((lons, lats), pm_values, (grid_x, grid_y), method=interpolation_method)


    print("\n[DIAGNOSTIC 3: Interpolated Data Range]")
    print(f"Min value after interpolation: {np.nanmin(grid_z):.2f}")
    print(f"Max value after interpolation: {np.nanmax(grid_z):.2f}")
    print("-" * 30)


    print("\nGenerating plot...")
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(map_image, extent=map_extent, origin='upper')


    vmin = df[pm_col_to_plot].quantile(0.05)
    vmax = df[pm_col_to_plot].quantile(0.95)

    print(f"Setting color bar range manually from {vmin:.2f} to {vmax:.2f} to ignore outliers.")

    contour = ax.contourf(
        grid_x, grid_y, grid_z,
        levels=15, cmap=cmap, alpha=alpha,
        vmin=vmin, vmax=vmax
    )

    cbar = fig.colorbar(contour, ax=ax, shrink=0.8)
    cbar.set_label(f'{pm_col_to_plot} Concentration (µg/m³)', fontsize=14) # Increased fontsize

    ax.set_title(f'{pm_col_to_plot} Concentration Heatmap', fontsize=16, weight='bold')
    ax.set_xlabel('Longitude', fontsize=14) # Increased fontsize
    ax.set_ylabel('Latitude', fontsize=14) # Increased fontsize
    ax.tick_params(axis='both', which='major', labelsize=12) # Increased tick label fontsize
    ax.set_aspect('equal', adjustable='box')

    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    print(f"\nSuccess! Heatmap saved as '{output_filename}'")
    plt.show()

if __name__ == '__main__':
    create_pm_heatmap()